# CUDA Matrix Multiplication (CPU vs GPU, nvprof)

SEED = 2096

Run this notebook on Google Colab with GPU enabled: Runtime > Change runtime type > GPU.

This notebook compiles `matmul.cu`, runs it for matrix sizes 256, 1024, and 4096, and profiles it.

In [1]:
!nvidia-smi

Tue Sep  1 20:52:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Upload matmul.cu

If running on Colab, upload `matmul.cu` from this repo (or mount Drive / clone the repo).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select matmul.cu

## Compile

In [ ]:
!nvcc -O3 matmul.cu -o matmul
# Note: -arch=native is recommended (targets the actual GPU directly, avoids the
# deprecated-architecture warning below) but this executed run used the plain command.

## Run for each matrix size

`matmul` takes N as a command-line argument and prints CPU time, GPU kernel time,
H2D+D2H transfer time, end-to-end GPU time, and speedup.

In [1]:
!./matmul 256

N=256
CPU time (ms): 20.3201
GPU kernel time (ms): 0.2337
H2D+D2H transfer time (ms): 0.3944
GPU end-to-end time (ms): 0.6281
Speedup (CPU / GPU end-to-end): 32.3519
Max abs diff vs CPU reference: 2.288818e-05


In [1]:
!./matmul 1024

N=1024
CPU time (ms): 3353.2892
GPU kernel time (ms): 7.8954
H2D+D2H transfer time (ms): 4.8241
GPU end-to-end time (ms): 12.7194
Speedup (CPU / GPU end-to-end): 263.6353
Max abs diff vs CPU reference: 9.155273e-05


In [1]:
!./matmul 4096

N=4096
CPU time (ms): 603595.4877
GPU kernel time (ms): 319.1948
H2D+D2H transfer time (ms): 76.1485
GPU end-to-end time (ms): 395.3433
Speedup (CPU / GPU end-to-end): 1526.7628
Max abs diff vs CPU reference: 3.662109e-04


## Profile

Try `nvprof` first (older Colab images); fall back to Nsight Systems if `nvprof` is unavailable
on the current CUDA version. State in METRICS.md which profiler was actually used.

In [1]:
!nvprof ./matmul 1024

==11737== NVPROF is profiling process 11737, command: ./matmul 1024
N=1024
CPU time (ms): 3359.2312
GPU kernel time (ms): 6.1646
H2D+D2H transfer time (ms): 5.0611
GPU end-to-end time (ms): 11.2257
Speedup (CPU / GPU end-to-end): 299.2448
Max abs diff vs CPU reference: 9.155273e-05
==11737== Profiling application: ./matmul 1024
==11737== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   65.87%  6.0313ms         1  6.0313ms  6.0313ms  6.0313ms  matmul_kernel(float const *, float const *, float*, int)
                   17.79%  1.6294ms         1  1.6294ms  1.6294ms  1.6294ms  [CUDA memcpy DtoH]
                   16.34%  1.4960ms         2  748.00us  744.47us  751.54us  [CUDA memcpy HtoD]
      API calls:   92.41%  173.98ms         3  57.994ms  56.581us  173.86ms  cudaMalloc
                    5.83%  10.986ms         3  3.6620ms  919.96us  9.0995ms  cudaMemcpy
                    1.32%  2.4829ms       114  21.779us   

In [ ]:
# Fallback if nvprof is not available on this CUDA version:
!nsys profile --stats=true ./matmul 1024

## Fill in METRICS.md

Copy the CPU time / GPU kernel time / transfer time / speedup numbers printed above into the
table in `METRICS.md` for N = 256, 1024, 4096. Paste the relevant profiler output (kernel time vs
transfer time breakdown) into `RUN_LOG.txt`.

## Crossover analysis

Across the three required sizes (256, 1024, 4096), the GPU already wins at the smallest one
tested: N=256 gives a 32x end-to-end speedup, so the actual crossover point is at or below 256.
It is not at size zero because every GPU run pays fixed costs regardless of matrix size — kernel
launch/API overhead and the H2D+D2H transfer (~0.4-0.6ms here) happen before any useful compute.
For a small enough matrix (well under 256 per side), the O(n^3) work the GPU parallelizes away
shrinks below that fixed cost, and a single CPU thread doing the same tiny amount of work would
win instead; at N=256 the compute already outweighs the fixed overhead, which is why GPU wins
there and beyond.